# Kata 24 — Slash Commands Custom y Skills

> Spec: [`specs/024-slash-commands-skills/spec.md`](../../specs/024-slash-commands-skills/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

Este kata es de **configuración de Claude Code**. Demuestro la mecánica creando los archivos reales y validando frontmatter.

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import pathlib, json, re, tempfile

client, settings = bootstrap(budget_calls=4)
print("modelo:", settings.model)


## 1. Concepto en mis palabras

Slash commands = trigger explícito (`/X`). Skills = pieza on-demand con
frontmatter que controla scope y tools. Project scope (`.claude/...`) se
comparte por git; user scope (`~/.claude/...`) es personal.

## 2. Por qué importa

Comandos de equipo en `~/.claude/commands/` no se replican al equipo.
Skills sin `context: fork` contaminan la sesión principal con output
verbose. Sin `allowed-tools` una skill puede borrar archivos por
accidente.

## 3. Crear archivos reales — slash command + skill

In [ ]:
workspace = pathlib.Path(tempfile.mkdtemp(prefix="kata24_"))

# Slash command project-scoped
cmd_path = workspace / ".claude/commands/review.md"
cmd_path.parent.mkdir(parents=True, exist_ok=True)
cmd_path.write_text("""Revisa el diff actual contra la guía de estilo.
Reporta findings tipados con severity y rule_id.""")

# Skill project-scoped con frontmatter
skill_path = workspace / ".claude/skills/codebase-analysis/SKILL.md"
skill_path.parent.mkdir(parents=True, exist_ok=True)
skill_path.write_text("""---
name: codebase-analysis
description: Mapea estructura de un módulo y devuelve resumen tipado.
context: fork
allowed-tools: ["Read", "Grep", "Glob"]
argument-hint: "<dir-or-feature>"
---
# Body de la skill

1. Glob `**/{argument}/**/*.{py,ts,tsx}` para listar archivos.
2. Grep imports cruzados.
3. Devuelve resumen con archivos clave y dependencias.
""")

print("workspace:", workspace)
for p in workspace.rglob("*.md"):
    print(" ", p.relative_to(workspace))

### 3.1 Validación del frontmatter de skill

In [ ]:
def parse_frontmatter(text: str) -> tuple[dict, str]:
    if not text.startswith("---\n"):
        return {}, text
    end = text.find("\n---", 4)
    if end < 0:
        return {}, text
    fm_text = text[4:end]
    body = text[end + 5:]
    fm = {}
    for line in fm_text.splitlines():
        if ":" not in line: continue
        k, _, v = line.partition(":")
        fm[k.strip()] = v.strip()
    return fm, body

fm, body = parse_frontmatter(skill_path.read_text())
print("frontmatter parseado:")
for k, v in fm.items():
    print(f"  {k}: {v}")

# Aserciones de salud
assert "name" in fm, "name es required"
assert fm.get("context") in {"fork", "shared"} or fm.get("context") is None, "context debe ser fork|shared"
print("\nOK: skill bien formada (context: fork → corre aislada)")

## 4. Anti-patrón — skill verbosa sin `context: fork`

In [ ]:
# Skill mal hecha: corre en sesión principal y devuelve 5000 tokens
bad_skill = workspace / ".claude/skills/discover-architecture/SKILL.md"
bad_skill.parent.mkdir(parents=True, exist_ok=True)
bad_skill.write_text("""---
name: discover-architecture
description: Explora arquitectura del proyecto. SIN context fork.
---
# Esta skill cargará todo el repo en context.
1. Glob **/*
2. Read cada uno
3. Reporta...
""")

fm_bad, _ = parse_frontmatter(bad_skill.read_text())
issues = []
if fm_bad.get("context") != "fork":
    issues.append("falta context: fork — la skill contaminará la sesión principal con discovery output")
if "allowed-tools" not in fm_bad:
    issues.append("falta allowed-tools — la skill puede invocar Write/Bash sin restricción")

print("issues detectados en bad_skill:")
for i in issues: print(" -", i)

### 4.1 Decisión rápida — slash command vs skill vs CLAUDE.md

In [ ]:
resp = client.messages.create(
    system="""Eres un asistente de Claude Code. Para cada necesidad, responde con UNA línea
indicando: 'CLAUDE.md', 'slash command', o 'skill', y por qué.
NO uses tools.""",
    messages=[{"role":"user","content":"""Decide para cada caso:

1. Cargar siempre la convención de "Python 3.11 + ruff" cuando el repo se abre.
2. Trigger manual: /generate-tests para crear tests del archivo abierto.
3. Análisis exploratorio que devuelve 8000 tokens de discovery.
4. Mi macro personal favorita: /commit-msg-fancy.
5. Restringir una operación de análisis a Read/Grep, sin Write."""}],
)
print(next((b.text for b in resp.content if b.type == "text"), ""))

## 5. Argumento de certificación

- **Slash commands**: `.claude/commands/X.md` (proyecto) vs
  `~/.claude/commands/X.md` (user). El equipo recibe los del proyecto al
  hacer git pull.
- **Skills frontmatter**:
  - `context: fork` → corre en sub-agente aislado, no contamina sesión.
  - `allowed-tools: [...]` → whitelist de tools (bloquea destructive).
  - `argument-hint: "..."` → texto que se pide cuando se invoca sin args.
- **CLAUDE.md** para convenciones siempre cargadas; **skill** para
  workflows on-demand.

## 6. Auto-evaluación

**1. Quiero un `/test-coverage` disponible cuando clonen el repo. ¿Dónde?**

`.claude/commands/test-coverage.md`. Project-scoped, versionado. Disponible
tras `git pull`.

**2. Una skill exploratoria que devuelve 5000 tokens de discovery: ¿qué
frontmatter aplica?**

`context: fork` (corre aislada) y `allowed-tools: ["Read","Grep","Glob"]`
(sin Write/Bash, exploración pura).

**3. ¿Cuándo prefiero CLAUDE.md sobre una skill?**

Cuando la convención debe estar **siempre cargada** (estilo, lenguaje,
arquitectura). Skill cuando el workflow es **on-demand** (review, audit,
análisis específico).